# MATLAB synthesis timing (§4.4)

Batch-run the MATLAB synthesis examples and record their solve times, step by step.

Per example we record:

- `design_s` — symbolic backstepping design (`reach_avoid_controller`)
- `sop_solve_s` — SOP bounded-control solve (`solvesop_bounded_control`)
- `synth_s` — `design_s + sop_solve_s` (the **offline** synthesis cost for the paper's timing table)
- `wall_s` — wall-clock of the whole `matlab -batch` subprocess
- `wall_net_s` — `wall_s` minus the measured MATLAB startup baseline

`design_s`/`sop_solve_s` come from `__TIMING__,<script>,<phase>,<seconds>` markers the example scripts print.
Reusable orchestration lives in `../python/matlab_runner.py`.

**Note:** running an example re-runs `export_to_python`, which writes a new timestamped controller into `../controllers/` (expected; the notebook-imported controllers have fixed timestamps and are not overwritten).


In [8]:
# ── Repo-root bootstrap: make ../python importable ──────────────────────────────
import sys, os

ROOT = os.getcwd()
for _ in range(4):
    if os.path.isdir(os.path.join(ROOT, "python")) and os.path.isdir(
        os.path.join(ROOT, "matlab")
    ):
        break
    ROOT = os.path.dirname(ROOT)
if os.path.join(ROOT, "python") not in sys.path:
    sys.path.insert(0, os.path.join(ROOT, "python"))
import importlib, matlab_runner as mr

importlib.reload(mr)
print("repo root:", ROOT)

repo root: /home/jianqiang/Downloads/reach_avoid_backstepping_MPC


## 1. Configuration

Edit the MATLAB binary, which examples to run, and how many repeats.


In [9]:
MATLAB_BIN = os.environ.get("MATLAB_BIN", "matlab")  # name on PATH or absolute path
MATLAB_DIR = os.path.join(ROOT, "matlab")
EXAMPLES = mr.DEFAULT_EXAMPLES  # or e.g. ["example_double_integrator"]
REPEATS = 1
TIMEOUT = 3600  # per-run seconds
MEASURE_BASELINE = True
OUT_CSV = os.path.join(ROOT, "data", "matlab_synthesis_timing.csv")
print("examples:", EXAMPLES)

examples: ['example_dubins_car', 'example_manipulator', 'example_double_integrator']


## 2. Resolve MATLAB and measure the startup baseline


In [10]:
matlab = mr.find_matlab(MATLAB_BIN)
assert (
    matlab
), f"MATLAB not found: {MATLAB_BIN!r} (set MATLAB_BIN env var or edit MATLAB_BIN)"
print("matlab:", matlab)

baseline = mr.measure_startup_baseline(matlab, TIMEOUT) if MEASURE_BASELINE else None
print("startup baseline (s):", None if baseline is None else round(baseline, 3))

matlab: /usr/local/bin/matlab
startup baseline (s): 3.787


## 3. Run each example (one row per run)

Each run launches `matlab -batch`, parses the timing markers, and reports status immediately.


In [11]:
rows = []
for ex in EXAMPLES:
    for rep in range(1, REPEATS + 1):
        print(f"running {ex} (run {rep}/{REPEATS}) ...", flush=True)
        res = mr.run_one(matlab, MATLAB_DIR, ex, TIMEOUT)
        res["repeat"] = rep
        res["wall_net_s"] = (res["wall_s"] - baseline) if baseline is not None else None
        rows.append(res)

        def _f(x):
            return "—" if x is None else f"{x:.3f}"

        print(
            f"  status={res['status']}  design={_f(res['design_s'])}s  "
            f"sop_solve={_f(res['sop_solve_s'])}s  synth={_f(res['synth_s'])}s  "
            f"wall={_f(res['wall_s'])}s"
        )
        if res["status"] != "ok":
            print("  error:", res["error"] or (res["stderr"] or "").strip()[-300:])

running example_dubins_car (run 1/1) ...
  status=ok  design=0.259s  sop_solve=64.337s  synth=64.596s  wall=69.741s
running example_manipulator (run 1/1) ...
  status=ok  design=0.321s  sop_solve=92.130s  synth=92.450s  wall=98.153s
running example_double_integrator (run 1/1) ...
  status=ok  design=0.183s  sop_solve=9.930s  synth=10.113s  wall=15.747s


## 4. Inspect the full MATLAB stdout of a run

Change the index to inspect any run's complete console output (solver reports, bounds, etc.).


In [12]:
idx = 0
print(rows[idx]["example"], "— stdout tail:\n")
print(rows[idx]["stdout"][-3000:])

example_dubins_car — stdout tail:

000067103*x1*x2^2 - 0.0000016528*x1*x2 - 0.0000034529*x1 + 0.000000092714*x2^4 + 0.0000035395*x2^3 + 0.00000013431*x2^2 - 0.000011602*x2 - 0.000001661))/302231454903657293676544 + (20675713704368153125*cos(th)*(0.000000063066*x1^4 + 0.00000025795*x1^3*x2 + 0.0000019413*x1^3 - 0.0000010165*x1^2*x2^2 - 0.00000067001*x1^2*x2 + 0.00000013078*x1^2 - 0.000000042555*x1*x2^3 + 0.0000011958*x1*x2^2 - 0.00000091138*x1*x2 - 0.0000064685*x1 + 0.0000012704*x2^4 - 0.00000056274*x2^3 + 0.0000045611*x2^2 + 0.0000022727*x2 - 0.0000012451))/302231454903657293676544 + 640*x1*cos(th) + 160*x2*sin(th) - 12500*v*cos(th)^2*(0.000000252264*x1^3 + 0.00000077385*x1^2*x2 + 0.0000058239*x1^2 - 0.000002033*x1*x2^2 - 0.00000134002*x1*x2 + 0.00000026156*x1 - 0.000000042555*x2^3 + 0.0000011958*x2^2 - 0.00000091138*x2 - 0.0000064685) - 12500*v*sin(th)^2*(- 0.0000018953*x1^3 - 0.00000054114*x1^2*x2 + 0.0000013745*x1^2 + 0.00000150153*x1*x2^2 + 0.00000134206*x1*x2 - 0.0000016528*x1 + 0

## 5. Summary table (mean over successful runs)


In [13]:
cols = [
    "example",
    "repeat",
    "status",
    "design_s",
    "sop_solve_s",
    "synth_s",
    "wall_s",
    "wall_net_s",
]
try:
    import pandas as pd

    df = pd.DataFrame([{k: r.get(k) for k in cols} for r in rows])
    display(df)
    ok = df[df.status == "ok"]
    if len(ok):
        display(
            ok.groupby("example")[
                ["design_s", "sop_solve_s", "synth_s", "wall_s"]
            ].mean()
        )
except ImportError:
    for r in rows:
        print({k: r.get(k) for k in cols})

,example,repeat,status,design_s,sop_solve_s,synth_s,wall_s,wall_net_s
0,example_dubins_car,1,ok,0.259344,64.336725,64.596069,69.741358,65.954839
1,example_manipulator,1,ok,0.320606,92.129594,92.450200,98.153294,94.366776
2,example_double_integrator,1,ok,0.183064,9.930109,10.113173,15.746574,11.960056


,design_s,sop_solve_s,synth_s,wall_s
example,,,,
example_double_integrator,0.183064,9.930109,10.113173,15.746574
example_dubins_car,0.259344,64.336725,64.596069,69.741358
example_manipulator,0.320606,92.129594,92.450200,98.153294


## 6. Write results to CSV


In [14]:
out = mr.write_csv(rows, OUT_CSV)
print("wrote", out)

wrote /home/jianqiang/Downloads/reach_avoid_backstepping_MPC/data/matlab_synthesis_timing.csv
